# Session 3: Neural Networks

In this exercise, we will finetune parameters for a Neural Network build to detect a DDoS attack.

The function "ddos_detection_NN" build the Neural Network with the parameters in the cell below it.


In the **first part** of this Notebook called "Data" we will download and briefly inspect the dataset we will work with.

In the **second part** called "Neural Network Architecture & Parameters" we will build and train a basic neural network. Your goal is it to adjust the parameters and optimize for model accuracy.

In [ ]:
import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import time

import random

import os

from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split

from sklearn.model_selection import StratifiedKFold

le = LabelEncoder()

# Data

In this chapter you don't need to edit anything, just download and briefly inspect the dataset we will be working with.

In [ ]:
# 5 MB
file_id = "1Bu6B0wUKglY1IAdsL361OkOGB5eJ2VTS"


url = f"https://drive.google.com/uc?export=download&id={file_id}"

data = pd.read_csv(url)

data.head()

This dataset contains network flow records with 319 features for each sample. Each row represents a single network flow, including details such as source and destination ports, flow duration, packet counts, payload statistics, and various statistical features for both forward and backward traffic directions. The dataset also includes categorical labels describing the flow’s nature, such as "Benign" or other activities.



For more infomration on the dataset, please refer to this [paper](https://www.mdpicom/2078-2489/15/4/195).

*Shafi, MohammadMoein, et al. "Toward generating a new cloud-based Distributed Denial of Service (DDoS) dataset and cloud intrusion traffic characterization." Information 15.4 (2024): 195.*

In [ ]:
data.fillna(value=np.nan, inplace=True)

data = data[(data.astype(str) != '?').all(axis=1)]

data = data.reset_index(drop = True)
# data = data.drop(columns=["flow_id", "timestamp", "src_ip", "dst_ip", "protocol"])
data.head()

In [ ]:
y = data.iloc[:,-2:-1]
y = y.where(y == "Benign", "Attack")
y.head()

In [ ]:
y = le.fit_transform(y)

In [ ]:
X = data.iloc[:,:-2]
X = np.array(X)


# Neural Network Architecture & Parameters

Here we build a neural network, train it with our data and evaluate it.

In [ ]:
import keras

from keras.models import Sequential

from keras.layers import Dense, Dropout, Activation

from keras import optimizers, regularizers

from sklearn import metrics

from keras.regularizers import l1, l2

from keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.model_selection import train_test_split

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn import metrics
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
from tensorflow.keras.layers import (
    Dense, Dropout,
    BatchNormalization, LeakyReLU
)

##  Parameters - Edit here

Start by editing these values here. They will be used to build the Neural Network and affect the accuracy of the final model.




In [ ]:
# ====== NN Hyperparameters (intentionally weak baseline for tuning) ======

# Regularization (too strong on purpose)
DROPOUT_RATE = 0.5       # Try: 0.00 to 0.50 (common 0.10 to 0.30)
L2_REG = 0.001            # Try: 0.00000 to 0.00100 (common 0.00001 to 0.00010)

# Network size (too small on purpose)
UNITS_1 = 16              # Try: 16 to 512
UNITS_2 = 8               # Try: 8 to 256
UNITS_3 = 4               # Try: 4 to 128

# Optimizer (slow learning on purpose)
LEARNING_RATE = 0.0001    # Try: 0.00010 to 0.00100

# Training process (more stable than batch 2)
BATCH_SIZE = 64           # Try: 8, 16, 32, 64, 128
EPOCHS = 100              # Try: 50 to 300
EARLY_STOP_PATIENCE = 5   # Try: 5 to 20
MIN_DELTA = 0.0001        # Try: 0.00001 to 0.00100

## Run

In [ ]:
def ddos_detection_NN(Xtrain, Xtest, ytrain, ytest):
    Xtrain = Xtrain.astype('float32')
    Xtest = Xtest.astype('float32')
    ytrain = ytrain.astype('float32')
    ytest = ytest.astype('float32')

    input_dim = Xtrain.shape[1]

    ddos_classifier = Sequential()
    ddos_classifier.add(BatchNormalization(input_shape=(input_dim,)))

    # Hidden layers
    ddos_classifier.add(Dense(units=UNITS_1, activation='relu', kernel_regularizer=regularizers.l2(L2_REG), input_dim=input_dim))
    ddos_classifier.add(Dropout(DROPOUT_RATE))
    ddos_classifier.add(Dense(units=UNITS_2, activation='relu', kernel_regularizer=regularizers.l2(L2_REG)))
    ddos_classifier.add(Dropout(DROPOUT_RATE))
    ddos_classifier.add(Dense(units=UNITS_3, activation='relu', kernel_regularizer=regularizers.l2(L2_REG)))
    ddos_classifier.add(Dropout(DROPOUT_RATE))

    ddos_classifier.add(Dense(units=1, activation='sigmoid'))

    adam = tf.keras.optimizers.Adam(LEARNING_RATE)
    ddos_classifier.compile(optimizer=adam, loss='binary_crossentropy', metrics=['accuracy'])

    monitor = EarlyStopping(
        monitor="val_loss",
        min_delta=MIN_DELTA,
        patience=EARLY_STOP_PATIENCE,
        verbose=0,
        mode='auto',
        restore_best_weights=True
    )
    callbacks_list = [monitor]

    history = ddos_classifier.fit(
        Xtrain, ytrain,
        validation_split=0.2,
        callbacks=callbacks_list,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE
    )

    pred_prob = ddos_classifier.predict(Xtest)
    pred = (pred_prob > 0.5)

    fbeta = metrics.fbeta_score(ytest, pred, average='weighted', beta=10)
    tn, fp, fn, tp = metrics.confusion_matrix(ytest, pred).ravel()
    tpr = tp / (tp + fn)
    fpr = fp / (fp + tn)

    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('model loss')
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend(['train', 'validation'], loc='upper left')
    plt.show()

    return fbeta, tpr, fpr


In [ ]:
# make the NN deterministic (for educational purposes)
os.environ['PYTHONHASHSEED'] = '1337'
random.seed(1337)
np.random.seed(1337)
tf.random.set_seed(1337)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
try:
    tf.config.experimental.enable_op_determinism()
except AttributeError:
    pass
#####################################################


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1337, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

cvscores_NN = []


start_time = time.time()

cvscores_NN.append(ddos_detection_NN(Xtrain = X_train, Xtest = X_test, ytrain=  y_train, ytest = y_test ))

print("--- %d minutes %d seconds ---" % (int((time.time() - start_time) // 60), int((time.time() - start_time) % 60)))

print("F-Beta Score", cvscores_NN[0][0])